In [2]:
# =============================================================================
# ETIQUETADO AUTOMÁTICO DE IMÁGENES POR VARIEDAD (Google Drive)
# Formato de salida: FECHA_ARA_[COD_VARIEDAD]_HOJA_001.jpg
# =============================================================================


# ==== CELDA 1: Montar Google Drive ====
from google.colab import drive
drive.mount('/content/drive')


# ==== CELDA 2: Imports ====
import os
import re
import shutil
from datetime import datetime


# ==== CELDA 3: Configuración ====

# Ruta base donde están tus carpetas por variedad
BASE_DIR = "/content/drive/MyDrive/Dataset_coffee_nuevo" #<- Cambiar la ruta sin es necesario

# Mapeo: nombre de carpeta -> código de variedad que quieres en el nombre
#Si tenemos otras variedades, debemos colocar el nombre de la carpeta de dicha variedad con su etiqueta, por ejemplo: "NombreCarpeta: Etiqueta" -> "Arabica_Caturra: CATU"

VARIEDADES = {
    "Arabica_Bourbon":  "BOU",
    "Arabica_Typica":   "TYP",
    "Arabica_Castillo": "CAST",
    "Arabica_Sarchimor": "SARMOR",
    "Arabica_Mundonovo": "MUNOVO"

}

# Extensiones de imagen válidas
EXTENSIONES_VALIDAS = (".jpg", ".jpeg", ".png")

# Patrón para extraer la fecha (AAAAMMDD) desde el nombre original,
# ej: IMG_20260728_155742_007.jpg -> 20260728
PATRON_FECHA = re.compile(r"(20\d{2}[01]\d[0-3]\d)")   # busca fecha, tipo 20YYMMDD

# Fecha de respaldo, SOLO se usa si un archivo no trae fecha reconocible
# en su nombre (por ejemplo si viene de otra fuente sin ese formato).
FECHA_RESPALDO = datetime.now().strftime("%Y%m%d")


def extraer_fecha_de_nombre(nombre_archivo):
    match = PATRON_FECHA.search(nombre_archivo)
    if match:
        return match.group(1)
    return FECHA_RESPALDO

# MODO = "renombrar" -> cambia el nombre directamente en la carpeta original
# (Arabica_Bourbon, Arabica_Typica, Arabica_Castillo), sin crear carpetas nuevas.
# ADVERTENCIA: esto es IRREVERSIBLE. Asegúrate de tener respaldo de tus fotos
# originales antes de ejecutar (ej. descarga un zip de la carpeta) por si algo
# sale mal a mitad de proceso.
MODO = "renombrar"   # "copiar" o "renombrar"
CARPETA_SALIDA = "/content/drive/MyDrive/Dataset_coffee_nuevo_etiquetado"


# ==== CELDA 4: Función de etiquetado ====

def etiquetar_variedad(carpeta_origen, codigo_variedad, modo, carpeta_salida=None):
    """
    Recorre una carpeta de imágenes de una variedad y genera nombres
    secuenciales tipo: FECHA_ARA_COD_HOJA_001.jpg
    La FECHA se extrae automáticamente del nombre original de cada imagen.
    """
    if not os.path.isdir(carpeta_origen):
        print(f"⚠ No existe la carpeta: {carpeta_origen}")
        return []

    # El orden alfabético ya respeta el orden cronológico porque el nombre
    # original empieza con AAAAMMDD_HHMMSS (ancho fijo) -> IMG_20260728_155742_007.jpg
    archivos = sorted([
        f for f in os.listdir(carpeta_origen)
        if f.lower().endswith(EXTENSIONES_VALIDAS)
    ])

    if not archivos:
        print(f"⚠ No se encontraron imágenes en: {carpeta_origen}")
        return []

    if modo == "copiar":
        destino_dir = os.path.join(carpeta_salida, os.path.basename(carpeta_origen))
        os.makedirs(destino_dir, exist_ok=True)

    registro = []
    for i, archivo in enumerate(archivos, start=1):
        ext = os.path.splitext(archivo)[1].lower()
        fecha_img = extraer_fecha_de_nombre(archivo)
        nuevo_nombre = f"{fecha_img}_ARA_{codigo_variedad}_HOJA_{i:03d}{ext}"

        ruta_original = os.path.join(carpeta_origen, archivo)

        if modo == "copiar":
            ruta_nueva = os.path.join(destino_dir, nuevo_nombre)
            shutil.copy2(ruta_original, ruta_nueva)
        elif modo == "renombrar":
            ruta_nueva = os.path.join(carpeta_origen, nuevo_nombre)
            os.rename(ruta_original, ruta_nueva)
        else:
            raise ValueError("MODO debe ser 'copiar' o 'renombrar'")

        registro.append({
            "nombre_original": archivo,
            "nombre_nuevo": nuevo_nombre,
            "fecha_extraida": fecha_img,
            "variedad": os.path.basename(carpeta_origen),
        })

    print(f"✔ {carpeta_origen} → {len(registro)} imágenes etiquetadas ({codigo_variedad})")
    return registro


# ==== CELDA 5: Ejecutar sobre las 3 variedades ====

registro_total = []

for nombre_carpeta, codigo in VARIEDADES.items():
    carpeta_origen = os.path.join(BASE_DIR, nombre_carpeta)
    resultado = etiquetar_variedad(
        carpeta_origen=carpeta_origen,
        codigo_variedad=codigo,
        modo=MODO,
        carpeta_salida=CARPETA_SALIDA,
    )
    registro_total.extend(resultado)

print(f"\nTotal general de imágenes etiquetadas: {len(registro_total)}")


# ==== CELDA 6: Ver el resultado en una tabla.
#<- Para ver la tabla hacemos clic en la opcion "Archivos" de la barra lateral izquierda y luego clic en "registro_etiquetado.csv"
import pandas as pd

df_registro = pd.DataFrame(registro_total)
df_registro.to_csv("/content/registro_etiquetado.csv", index=False)
df_registro

Mounted at /content/drive
✔ /content/drive/MyDrive/Dataset_coffee_nuevo/Arabica_Bourbon → 267 imágenes etiquetadas (BOU)
✔ /content/drive/MyDrive/Dataset_coffee_nuevo/Arabica_Typica → 129 imágenes etiquetadas (TYP)
✔ /content/drive/MyDrive/Dataset_coffee_nuevo/Arabica_Castillo → 171 imágenes etiquetadas (CAST)
✔ /content/drive/MyDrive/Dataset_coffee_nuevo/Arabica_Sarchimor → 229 imágenes etiquetadas (SARMOR)
✔ /content/drive/MyDrive/Dataset_coffee_nuevo/Arabica_Mundonovo → 305 imágenes etiquetadas (MUNOVO)

Total general de imágenes etiquetadas: 1101


,nombre_original,nombre_nuevo,fecha_extraida,variedad
0,20260728_ARA_BOU_HOJA_001.jpg,20260728_ARA_BOU_HOJA_001.jpg,20260728,Arabica_Bourbon
1,20260728_ARA_BOU_HOJA_002.jpg,20260728_ARA_BOU_HOJA_002.jpg,20260728,Arabica_Bourbon
2,20260728_ARA_BOU_HOJA_003.jpg,20260728_ARA_BOU_HOJA_003.jpg,20260728,Arabica_Bourbon
3,20260728_ARA_BOU_HOJA_004.jpg,20260728_ARA_BOU_HOJA_004.jpg,20260728,Arabica_Bourbon
4,20260728_ARA_BOU_HOJA_005.jpg,20260728_ARA_BOU_HOJA_005.jpg,20260728,Arabica_Bourbon
...,...,...,...,...
1096,20260803_ARA_MUNOVO_HOJA_301.jpg,20260803_ARA_MUNOVO_HOJA_301.jpg,20260803,Arabica_Mundonovo
1097,20260803_ARA_MUNOVO_HOJA_302.jpg,20260803_ARA_MUNOVO_HOJA_302.jpg,20260803,Arabica_Mundonovo
1098,20260803_ARA_MUNOVO_HOJA_303.jpg,20260803_ARA_MUNOVO_HOJA_303.jpg,20260803,Arabica_Mundonovo
1099,20260803_ARA_MUNOVO_HOJA_304.jpg,20260803_ARA_MUNOVO_HOJA_304.jpg,20260803,Arabica_Mundonovo
